# 🤖 Josh Kenya LLM - Google Colab Training Notebook

## Train a 50M Parameter Language Model on Free GPU

This notebook sets up and trains Josh Kenya LLM on Google Colab. The entire process takes about 1.5-2 hours on a free T4 GPU.

### Setup Instructions:
1. Click **Runtime** → **Change runtime type** → Select **GPU (T4)**
2. Run each cell in order
3. Download your trained model when done!

## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q datasets transformers tqdm pyyaml flask flask-cors

## Step 2: Clone Repository and Setup

In [ ]:
# Clone the repository
!git clone https://github.com/aipulse54-svg/Llm_kenya.git
%cd Llm_kenya

# Verify files
!ls -la

## Step 3: Setup Python Path and Imports

In [ ]:
import sys
import os

# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU not available! Go to Runtime → Change runtime type and select GPU")

## Step 4: Verify Model Architecture

In [ ]:
from model_architecture import JoshKenyaLLM
import yaml

# Load config
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Create model
model = JoshKenyaLLM(config['model'])
num_params = model.get_num_params()

print("="*60)
print("Josh Kenya LLM Model Architecture".center(60))
print("="*60)
print(f"Total Parameters: {num_params:,} ({num_params/1e6:.1f}M)")
print(f"\nModel Configuration:")
print(f"  Hidden Size: {config['model']['hidden_size']}")
print(f"  Num Layers: {config['model']['num_hidden_layers']}")
print(f"  Attention Heads: {config['model']['num_attention_heads']}")
print(f"  Intermediate Size: {config['model']['intermediate_size']}")
print(f"  Vocab Size: {config['model']['vocab_size']}")
print(f"  Max Position Embeddings: {config['model']['max_position_embeddings']}")
print("="*60)

## Step 5: Customize Training Config (Optional)

Modify settings below if needed:

In [ ]:
# Optional: Customize training parameters
# If you have less VRAM, reduce batch_size

config['training']['batch_size'] = 32  # Reduce to 16 if running out of memory
config['training']['num_epochs'] = 1   # Start with 1 epoch for testing
config['training']['learning_rate'] = 5e-4
config['model']['max_position_embeddings'] = 1024  # Reduce if memory issues

# Save updated config
with open('config.yaml', 'w') as f:
    yaml.dump(config, f)

print("✓ Training config updated")
print(f"\nTraining Settings:")
print(f"  Batch Size: {config['training']['batch_size']}")
print(f"  Learning Rate: {config['training']['learning_rate']}")
print(f"  Num Epochs: {config['training']['num_epochs']}")
print(f"  Max Seq Length: {config['model']['max_position_embeddings']}")

## Step 6: Start Training 🚀

This will train the model on WikiText-2 dataset. Progress will be shown below.

In [ ]:
from trainer import LLMTrainer

print("\n" + "="*60)
print("Josh Kenya LLM - Training Pipeline".center(60))
print("="*60 + "\n")

try:
    # Initialize trainer
    trainer = LLMTrainer(config_path='config.yaml', device='cuda')
    
    print(f"✓ Model initialized with {trainer.model.get_num_params():,} parameters")
    print(f"✓ Using device: {trainer.device}")
    print(f"\nStarting training...\n")
    
    # Start training
    trainer.train()
    
    print("\n" + "="*60)
    print("✓ Training completed successfully!".center(60))
    print("="*60)
    print(f"\n✓ Model saved to: models/josh_kenya_lm.pt")
    print(f"✓ Config saved to: models/config.yaml")
    print(f"✓ Checkpoints in: checkpoints/")
    
except KeyboardInterrupt:
    print("\n\nTraining interrupted by user")
except Exception as e:
    print(f"\nError during training: {e}")
    import traceback
    traceback.print_exc()

## Step 7: Verify Model Files

In [ ]:
import os
from pathlib import Path

print("Model files:")
if os.path.exists('models'):
    for file in os.listdir('models'):
        path = os.path.join('models', file)
        size = os.path.getsize(path) / 1e6  # Size in MB
        print(f"  ✓ {file} ({size:.1f} MB)")
else:
    print("  No models directory found")

print("\nCheckpoint files:")
if os.path.exists('checkpoints'):
    for file in sorted(os.listdir('checkpoints')):
        path = os.path.join('checkpoints', file)
        size = os.path.getsize(path) / 1e6
        print(f"  ✓ {file} ({size:.1f} MB)")
else:
    print("  No checkpoints directory found")

## Step 8: Test Inference (Optional)

In [ ]:
from inference import JoshKenyaInference

print("Loading trained model...")
try:
    model = JoshKenyaInference(model_dir='models', device='cuda')
    
    # Show model info
    info = model.get_model_info()
    print(f"\nModel Information:")
    print(f"  Name: {info['name']}")
    print(f"  Parameters: {info['parameters_millions']}")
    print(f"  Device: {info['device']}")
    
    # Test generation
    print(f"\n{'='*60}")
    print("Testing model generation...".center(60))
    print('='*60)
    
    question = "What is artificial intelligence?"
    print(f"\nQuestion: {question}")
    print(f"\nJosh Kenya: ", end="", flush=True)
    
    response = model.answer_question(question)
    print(response)
    
    print(f"\n{'='*60}")
    print("✓ Model is working!".center(60))
    print('='*60)
    
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

## Step 9: Download Trained Model

In [ ]:
# Create a zip file with the trained model
import shutil
import os

print("Creating download package...")

# Zip the models directory
if os.path.exists('models'):
    shutil.make_archive('josh_kenya_trained_model', 'zip', '.', 'models')
    size = os.path.getsize('josh_kenya_trained_model.zip') / 1e6
    print(f"✓ Created: josh_kenya_trained_model.zip ({size:.1f} MB)")
    print("\nYou can download this file from the Files menu on the left")
    print("\nTo use the trained model:")
    print("  1. Download josh_kenya_trained_model.zip")
    print("  2. Unzip it in your project directory")
    print("  3. Run: python src/app.py")
else:
    print("⚠️ Models directory not found. Train the model first.")

## Step 10: Run Web UI (Optional - for testing)

Note: Web UI won't fully work in Colab, but you can test inference

In [ ]:
# Quick chat test
from inference import JoshKenyaInference

print("Starting interactive chat...")
print("Type 'exit' to quit\n")

try:
    model = JoshKenyaInference(model_dir='models', device='cuda')
    
    # Test a few questions
    questions = [
        "What is machine learning?",
        "Tell me about neural networks",
        "How do transformers work?"
    ]
    
    print(f"{'='*60}")
    print("Josh Kenya LLM Chat Test".center(60))
    print('='*60)
    
    for question in questions:
        print(f"\nYou: {question}")
        response = model.answer_question(question)
        print(f"Josh Kenya: {response}")
        print("-" * 60)
    
    # Show conversation history
    history = model.memory.get_history()
    print(f"\nConversation memory: {len(history)} turns")
    
except Exception as e:
    print(f"Error: {e}")

## Summary

✅ **What you've accomplished:**

1. **Trained a 50M parameter LLM** on WikiText-2 dataset
2. **Model saved** with all parameters and config
3. **Tested inference** - model can answer questions
4. **Ready to download** - use locally with web UI

### Next Steps:

1. **Download the trained model:**
   - `josh_kenya_trained_model.zip`

2. **Use it locally:**
   ```bash
   unzip josh_kenya_trained_model.zip
   python src/app.py
   ```

3. **Open web UI:**
   - http://localhost:5000

### Training Tips:

- **Out of memory?** Reduce `batch_size` to 16
- **Longer training?** Increase `num_epochs` to 2-3
- **Faster testing?** Reduce `max_position_embeddings` to 512
- **Better results?** Train for more epochs

### Files in models/ directory:

- `josh_kenya_lm.pt` - Trained model weights
- `config.yaml` - Model configuration
- Use with `JoshKenyaInference` class

---

**Congratulations on training Josh Kenya LLM! 🚀**